# SQL → pandas: Tradução prática
Este notebook demonstra como reproduzir em **pandas** as operações SQL mais comuns
(SELECT, WHERE, ORDER BY, DISTINCT, GROUP BY/HAVING, JOINs, UNION, subqueries,
CTEs, CASE, funções de string e data, tratamento de NULL, e **funções de janela** como `ROW_NUMBER`, `RANK`, `LAG`/`LEAD`).

**Datasets** usados (gerados sinteticamente): `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`, `payments.csv`.

> Dica: pense em cada `DataFrame` como uma _tabela_. Encadeie operações como se fossem subconsultas/CTEs.


In [3]:
import os
import pandas as pd, numpy as np

pd.set_option('display.max_rows', 8)
customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])
products  = pd.read_csv('data/products.csv')
orders    = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv('data/order_items.csv')
payments    = pd.read_csv('data/payments.csv', parse_dates=['paid_at'])

customers.head(), products.head(), orders.head()

(   customer_id         name                   email          city state  \
 0            1  Cliente 001  cliente001@exemplo.com     São Paulo    PR   
 1            2  Cliente 002  cliente002@exemplo.com      Brasília    SP   
 2            3  Cliente 003  cliente003@exemplo.com     Fortaleza    RS   
 3            4  Cliente 004  cliente004@exemplo.com  Porto Alegre    CE   
 4            5  Cliente 005  cliente005@exemplo.com  Porto Alegre    RS   
 
   signup_date  
 0  2024-12-28  
 1  2024-05-01  
 2  2022-12-17  
 3  2024-11-30  
 4  2023-07-01  ,
    product_id         name     category    price  active
 0           1  Produto 001         Moda  1397.54    True
 1           2  Produto 002       Livros   188.40    True
 2           3  Produto 003       Livros   184.48   False
 3           4  Produto 004  Eletrônicos   140.69    True
 4           5  Produto 005       Livros   990.22   False,
    order_id  customer_id order_date     status
 0         1          185 2024-08-26  canc

## 1) SELECT / WHERE / ORDER BY / LIMIT
**SQL**
```sql
SELECT name, city, state 
FROM customers 
WHERE state = 'SP'
ORDER BY name ASC
OFFSET 0 ROWS FETCH NEXT 5 ROWS ONLY;
```
**pandas**


In [4]:
(customers.loc[customers['state']=='SP', ['name','city','state']]
          .sort_values('name')
          .head(5))

,name,city,state
1,Cliente 002,Brasília,SP
7,Cliente 008,Fortaleza,SP
16,Cliente 017,Recife,SP
17,Cliente 018,Rio de Janeiro,SP
19,Cliente 020,Porto Alegre,SP


## 2) DISTINCT
**SQL**
```sql
SELECT DISTINCT category FROM products;
```
**pandas**


In [24]:
products[['category']].drop_duplicates()

,category
0,Moda
1,Livros
3,Eletrônicos
5,Beleza
6,Esporte
8,Brinquedos
16,Casa


## 3) Agregações, GROUP BY e HAVING
**SQL**
```sql
SELECT category, COUNT(*) AS qtde, ROUND(AVG(price),2) AS preco_medio
FROM products
GROUP BY category
HAVING COUNT(*) > 10
ORDER BY qtde DESC;
```
**pandas**


In [6]:
(products
 .groupby('category', as_index=False)
 .agg(qtde=('product_id','count'), preco_medio=('price','mean'))
 .query('qtde > 10')
 .sort_values('qtde', ascending=False)
 .assign(preco_medio=lambda df: df['preco_medio'].round(2)))

,category,qtde,preco_medio
0,Beleza,23,671.01
5,Livros,22,807.29
6,Moda,20,740.88
1,Brinquedos,15,734.58
2,Casa,14,767.06
3,Eletrônicos,14,695.67
4,Esporte,12,998.50


## 4) JOINs (INNER, LEFT, FULL OUTER)
**SQL**
```sql
SELECT o.order_id, c.name, o.order_date
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id;
```
**pandas** – `merge`


In [7]:
inner = orders.merge(customers[['customer_id','name']], on='customer_id', how='inner')
left  = orders.merge(customers[['customer_id','name']], on='customer_id', how='left')
full  = orders.merge(customers[['customer_id','name']], on='customer_id', how='outer')
inner.head(), left.head(), full.head()

(   order_id  customer_id order_date     status         name
 0         1          185 2024-08-26  cancelado  Cliente 185
 1         2          198 2023-01-08       pago  Cliente 198
 2         3          192 2023-01-23       pago  Cliente 192
 3         4          112 2024-05-24  cancelado  Cliente 112
 4         5           41 2023-08-18  cancelado  Cliente 041,
    order_id  customer_id order_date     status         name
 0         1          185 2024-08-26  cancelado  Cliente 185
 1         2          198 2023-01-08       pago  Cliente 198
 2         3          192 2023-01-23       pago  Cliente 192
 3         4          112 2024-05-24  cancelado  Cliente 112
 4         5           41 2023-08-18  cancelado  Cliente 041,
    order_id  customer_id order_date     status         name
 0     157.0            1 2024-08-25  cancelado  Cliente 001
 1     214.0            1 2024-03-30  cancelado  Cliente 001
 2     329.0            1 2024-05-19    enviado  Cliente 001
 3     402.0          

## 5) UNION vs UNION ALL
**SQL**
```sql
SELECT city FROM customers
UNION
SELECT state FROM customers;
```
`UNION ALL` mantém duplicatas.

**pandas**


In [ ]:
union_all = pd.concat([customers[['city']].rename(columns={'city':'val'}),
                                  customers[['state']].rename(columns={'state':'val'})],
                                 ignore_index=True)
union     = union_all.drop_duplicates()
#union_all.head(8), union.head(8)

(            val
 0     São Paulo
 1      Brasília
 2     Fortaleza
 3  Porto Alegre
 4  Porto Alegre
 5      Salvador
 6     São Paulo
 7     Fortaleza,
                val
 0        São Paulo
 1         Brasília
 2        Fortaleza
 3     Porto Alegre
 5         Salvador
 8   Belo Horizonte
 10          Recife
 11        Campinas)

## 6) Subconsultas
**SQL**
```sql
SELECT * FROM orders
WHERE customer_id IN (SELECT customer_id FROM customers WHERE state='SP');
```
**pandas**


In [9]:
sp_customers = customers.loc[customers['state']=='SP','customer_id']
orders[orders['customer_id'].isin(sp_customers)].head()

,order_id,customer_id,order_date,status
0,1,185,2024-08-26,cancelado
12,13,169,2024-02-25,pago
13,14,79,2023-09-10,cancelado
15,16,17,2024-03-12,finalizado
17,18,152,2023-04-06,enviado


## 7) CTE (WITH)
**SQL**
```sql
WITH por_cliente AS (
  SELECT customer_id, COUNT(*) AS qtd FROM orders GROUP BY customer_id
)
SELECT * FROM por_cliente WHERE qtd >= 3;
```
**pandas** – crie um DataFrame intermediário:


In [10]:
por_cliente = (orders.groupby('customer_id', as_index=False)
                      .agg(qtd=('order_id','count')))
por_cliente[por_cliente['qtd'] >= 3].head()

,customer_id,qtd
0,1,7
1,2,4
3,4,4
4,5,3
5,6,4


## 8) CASE WHEN
**SQL**
```sql
SELECT order_id,
       CASE 
         WHEN amount >= 1000 THEN 'alto'
         WHEN amount >= 200 THEN 'medio'
         ELSE 'baixo'
       END AS faixa
FROM payments;
```
**pandas** – `np.select`


In [11]:
condicoes = [payments['amount']>=1000, payments['amount']>=200]
valores = ['alto','medio']
payments.assign(faixa=np.select(condicoes, valores, default='baixo')).head()

,payment_id,order_id,method,amount,paid_at,faixa
0,1,45,pix,2937.35,2023-12-17,alto
1,2,78,cartao_credito,2681.99,2023-01-15,alto
2,3,226,boleto,416.12,2024-06-21,medio
3,4,488,voucher,2637.58,2023-03-18,alto
4,5,96,pix,2519.14,2024-09-24,alto


## 9) NULLs e COALESCE/ISNULL
**SQL**
```sql
SELECT COALESCE(email,'sem_email') FROM customers;
```
**pandas** – `fillna`


In [12]:
customers.assign(email_coalescido=customers['email'].fillna('sem_email')).head()

,customer_id,name,email,city,state,signup_date,email_coalescido
0,1,Cliente 001,cliente001@exemplo.com,São Paulo,PR,2024-12-28,cliente001@exemplo.com
1,2,Cliente 002,cliente002@exemplo.com,Brasília,SP,2024-05-01,cliente002@exemplo.com
2,3,Cliente 003,cliente003@exemplo.com,Fortaleza,RS,2022-12-17,cliente003@exemplo.com
3,4,Cliente 004,cliente004@exemplo.com,Porto Alegre,CE,2024-11-30,cliente004@exemplo.com
4,5,Cliente 005,cliente005@exemplo.com,Porto Alegre,RS,2023-07-01,cliente005@exemplo.com


## 10) Funções de string
**SQL**
```sql
SELECT UPPER(name), SUBSTRING(email,1,5), CONCAT(city, '-', state) FROM customers;
```
**pandas**


In [13]:
tmp = customers.copy()
tmp['name_up'] = tmp['name'].str.upper()
tmp['email_sub'] = tmp['email'].fillna('').str[:5]
tmp['cidade_uf'] = tmp['city'].str.cat(tmp['state'], sep='-')
tmp[['name_up','email_sub','cidade_uf']].head()

,name_up,email_sub,cidade_uf
0,CLIENTE 001,clien,São Paulo-PR
1,CLIENTE 002,clien,Brasília-SP
2,CLIENTE 003,clien,Fortaleza-RS
3,CLIENTE 004,clien,Porto Alegre-CE
4,CLIENTE 005,clien,Porto Alegre-RS


## 11) Funções de data
**SQL**
```sql
SELECT YEAR(order_date) ano, MONTH(order_date) mes, COUNT(*) qtd
FROM orders
GROUP BY YEAR(order_date), MONTH(order_date);
```
**pandas**


In [14]:
(orders
 .assign(ano=lambda df: df['order_date'].dt.year,
         mes=lambda df: df['order_date'].dt.month)
 .groupby(['ano','mes'], as_index=False)
 .agg(qtd=('order_id','count'))
 .sort_values(['ano','mes']).head(12))

,ano,mes,qtd
0,2023,1,26
1,2023,2,20
2,2023,3,34
3,2023,4,32
...,...,...,...
8,2023,9,34
9,2023,10,34
10,2023,11,21
11,2023,12,26


## 12) Funções de janela (window functions)
**SQL (exemplos)**
```sql
SELECT order_id,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) rn,
       RANK()       OVER (PARTITION BY customer_id ORDER BY order_date) rk,
       LAG(amount)  OVER (PARTITION BY order_id ORDER BY paid_at) lag_amt
FROM payments p JOIN orders o USING(order_id);
```
**pandas** – `groupby().cumcount()`, `rank()`, `shift()`, `transform()`


In [15]:
# Preparar pagamentos com datas ordenadas
p = payments.merge(orders[['order_id','customer_id','order_date']], on='order_id', how='left').sort_values(['customer_id','order_date','paid_at'])
p['row_number'] = p.groupby('customer_id').cumcount() + 1
p['rank_in_customer'] = p.groupby('customer_id')['order_date'].rank(method='dense')
p['prev_amount_by_order'] = p.groupby('order_id')['amount'].shift(1)   # LAG por order_id
p[['order_id','customer_id','order_date','paid_at','amount','row_number','rank_in_customer','prev_amount_by_order']].head(10)

,order_id,customer_id,order_date,paid_at,amount,row_number,rank_in_customer,prev_amount_by_order
391,402,1,2023-02-28,2023-12-20,2280.49,1,1.0,NaN
329,329,1,2024-05-19,2023-07-23,2072.65,2,2.0,NaN
438,470,1,2024-06-30,2023-01-20,1541.10,3,3.0,NaN
90,157,1,2024-08-25,2023-03-04,272.82,4,4.0,NaN
...,...,...,...,...,...,...,...,...
419,110,2,2024-04-27,2024-05-15,1733.54,3,2.0,NaN
191,452,2,2024-09-16,2023-06-30,2254.95,4,3.0,NaN
279,452,2,2024-09-16,2023-08-02,2896.14,5,3.0,2254.95
332,41,3,2023-02-26,2023-01-20,2294.38,1,1.0,NaN


## 13) Agregações com JOIN (receita por cliente)
**SQL**
```sql
SELECT c.customer_id, c.name, SUM(oi.quantity*oi.unit_price) AS receita
FROM customers c
JOIN orders o   ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.customer_id, c.name
ORDER BY receita DESC;
```
**pandas**


In [16]:
itens_val = order_items.assign(item_total = order_items['quantity']*order_items['unit_price'])
receita = (orders.merge(itens_val[['order_id','item_total']], on='order_id', how='left')
                .merge(customers[['customer_id','name']], on='customer_id', how='left')
                .groupby(['customer_id','name'], as_index=False)
                .agg(receita=('item_total','sum'))
                .fillna({'receita':0})
                .sort_values('receita', ascending=False))
receita.head(10)

,customer_id,name,receita
0,1,Cliente 001,72926.34
29,31,Cliente 031,54591.17
56,60,Cliente 060,52716.49
73,79,Cliente 079,50104.94
...,...,...,...
162,171,Cliente 171,44240.17
184,193,Cliente 193,42928.61
173,182,Cliente 182,41993.91
113,119,Cliente 119,39723.87


## 14) HAVING com condição de agregação
Top 5 categorias por **ticket médio** (> R$ 300).


In [17]:
ticket = (order_items.merge(products[['product_id','category']], on='product_id', how='left')
                    .assign(total=lambda df: df['quantity']*df['unit_price'])
                    .groupby('category', as_index=False)
                    .agg(ticket_medio=('total','mean'))
                   )
ticket.query('ticket_medio > 300').sort_values('ticket_medio', ascending=False).head(5)

,category,ticket_medio
2,Casa,2430.256649
4,Esporte,2276.615057
5,Livros,2204.381439
0,Beleza,2179.695945
6,Moda,2116.761538


## 15) Anti-join (NOT IN / LEFT JOIN WHERE NULL)
**SQL**
```sql
SELECT o.*
FROM orders o
LEFT JOIN payments p ON p.order_id = o.order_id
WHERE p.order_id IS NULL;
```
**pandas**


In [18]:
o = orders[['order_id','customer_id','order_date','status']]
paid_ids = payments['order_id'].unique()
o[~o['order_id'].isin(paid_ids)].head()

,order_id,customer_id,order_date,status
4,5,41,2023-08-18,cancelado
7,8,199,2023-09-19,finalizado
10,11,101,2024-05-09,pago
12,13,169,2024-02-25,pago
13,14,79,2023-09-10,cancelado


## 16) UPDATE (imputação) e DELETE (filtro)
**SQL**
```sql
UPDATE order_items SET unit_price = 0 WHERE unit_price IS NULL;
DELETE FROM orders WHERE status='cancelado';
```
**pandas**


In [19]:
# "UPDATE": criar uma cópia com imputação
order_items_upd = order_items.copy()
order_items_upd['unit_price'] = order_items_upd['unit_price'].fillna(0)

# "DELETE": apenas filtre fora
orders_sem_cancelados = orders[orders['status']!='cancelado']
order_items_upd.head(), orders_sem_cancelados['status'].unique()

(   order_item_id  order_id  product_id  quantity  unit_price
 0              1        25          57         5     1027.01
 1              2       142          36         2      967.66
 2              3       338         101         2      982.42
 3              4        30           8         5      916.11
 4              5       214          57         4     1439.06,
 array(['pago', 'finalizado', 'enviado', 'novo'], dtype=object))

## 17) COUNT(DISTINCT ...) e percentuais (window / transform)

**SQL**
```sql
-- 17.1) COUNT(DISTINCT ...) por estado
SELECT state, COUNT(DISTINCT customer_id) AS clientes_unicos
FROM customers
GROUP BY state
ORDER BY clientes_unicos DESC;

-- 17.2) Percentual de cada pagamento no total do cliente (window)
SELECT p.order_id, p.amount, o.customer_id,
       p.amount / SUM(p.amount) OVER (PARTITION BY o.customer_id) AS pct_no_cliente
FROM payments p
JOIN orders o ON o.order_id = p.order_id;


In [9]:
# 17.2) Trazer customer_id para payments via merge com orders
payments_w_customer = payments.merge(
    orders[['order_id', 'customer_id']],
    on='order_id',
    how='left'
)

payments_pct = payments_w_customer.assign(
    pct_no_cliente=lambda df: df['amount'] / df.groupby('customer_id')['amount'].transform('sum')
)

payments_pct[['customer_id', 'order_id', 'amount', 'pct_no_cliente']].head()


,customer_id,order_id,amount,pct_no_cliente
0,169,45,2937.35,0.471910
1,171,78,2681.99,0.178080
2,135,226,416.12,0.066198
3,121,488,2637.58,0.364411
4,68,96,2519.14,0.633335


## 18) PIVOT e UNPIVOT (relatórios)

**SQL**

- A ideia do PIVOT é transformar valores em colunas (ex.: status vira coluna).
- A sintaxe exata muda por banco (SQL Server/Oracle/Postgres etc.).
- Exemplo conceitual: contar pedidos por status por cliente.
- (Em muitos bancos seria feito com PIVOT ou com SUM(CASE WHEN ...))
- UNPIVOT faz o inverso: colunas voltam a virar linhas.


In [10]:
# 18.1) PIVOT: pedidos por status (status vira colunas)
pivot_status = (
    orders
      .pivot_table(
          index='customer_id',
          columns='status',
          values='order_id',
          aggfunc='count',
          fill_value=0
      )
      .reset_index()
)
pivot_status.head()

# 18.2) UNPIVOT (melt): colunas de status voltam a virar linhas
unpivot_status = pivot_status.melt(
    id_vars=['customer_id'],
    var_name='status',
    value_name='qtd_pedidos'
)
unpivot_status.head()


,customer_id,status,qtd_pedidos
0,1,cancelado,2
1,2,cancelado,1
2,3,cancelado,0
3,4,cancelado,0
4,5,cancelado,1


## 19) TOP N por grupo (ROW_NUMBER + filtro)

Exemplo: pegar os **3 pedidos mais recentes de cada cliente**.

**SQL**
```sql
SELECT *
FROM (
  SELECT o.*,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn
  FROM orders o
) t
WHERE rn <= 3
ORDER BY customer_id, order_date DESC;


In [8]:
top3_por_cliente = (
    orders
      .sort_values(['customer_id', 'order_date'], ascending=[True, False])
      .assign(rn=lambda df: df.groupby('customer_id').cumcount() + 1)
      .query('rn <= 3')
      .sort_values(['customer_id', 'order_date'], ascending=[True, False])
)
top3_por_cliente.head(15)


,order_id,customer_id,order_date,status,rn
156,157,1,2024-08-25,cancelado,1
469,470,1,2024-06-30,finalizado,2
328,329,1,2024-05-19,enviado,3
451,452,2,2024-09-16,enviado,1
...,...,...,...,...,...
86,87,5,2023-11-19,pago,2
448,449,5,2023-02-11,enviado,3
412,413,6,2024-05-30,finalizado,1
509,510,6,2024-01-11,finalizado,2
